# Rüzgar Colab Kurulumu

Bu defter:
- `umithaymana/ruzgar` deposunu klonlar
- uygun `requirements` dosyalarını kurar
- GPU durumunu kontrol eder
- `main.py` çalıştırma komutunu hazırlar

In [ ]:
# 1) Depoyu klonla
import os, shutil, subprocess, textwrap

REPO_URL = "https://github.com/umithaymana/ruzgar.git"
REPO_DIR = "/content/ruzgar"

if os.path.exists(REPO_DIR):
    print(f"Temizleniyor: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

print("Klonlanıyor...")
subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
print("Tamam.")

%cd /content/ruzgar

In [ ]:
# 2) Gerekli kütüphaneleri kur
# Not: ilim-voice/requirements.txt içindeki TTS (Coqui) Colab'da sık hata verir.
# Repoda ilim-voice/requirements-colab.txt var — TTS satırı yok.
import os, subprocess

def build_req_list():
    out = [
        "ilim-assistant/requirements.txt",
        "ilim-video/requirements.txt",
    ]
    vcol = "ilim-voice/requirements-colab.txt"
    vfull = "ilim-voice/requirements.txt"
    if os.path.exists(vcol):
        out.append(vcol)
    elif os.path.exists(vfull):
        out.append(vfull)
    return [p for p in out if os.path.exists(p)]

reqs = build_req_list()
if not reqs:
    raise FileNotFoundError("requirements dosyası bulunamadı.")

subprocess.check_call(["python", "-m", "pip", "install", "--upgrade", "pip"])

for req in reqs:
    print(f"\nKuruluyor: {req}")
    subprocess.check_call(["python", "-m", "pip", "install", "-r", req])

print("\nKurulum tamam.")

In [ ]:
# 3) GPU kontrolü
import subprocess, torch

print("CUDA kullanılabilir mi?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU görünmüyor. Colab Runtime > Change runtime type > GPU seçin.")

print("\nNVIDIA-SMI çıktısı:")
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as e:
    print("nvidia-smi çalıştırılamadı:", e)

In [ ]:
# 4) main.py başlatmaya hazır hale getir
import os

MAIN_PATH = "main.py"
ALT_PATHS = [
    "ilim-assistant/desktop_server.py",
    "ilim-assistant/gradio_chat.py",
]

if os.path.exists(MAIN_PATH):
    RUN_CMD = f"python {MAIN_PATH}"
    print("Hazır komut:", RUN_CMD)
else:
    print("main.py bulunamadı.")
    found = [p for p in ALT_PATHS if os.path.exists(p)]
    if found:
        print("Projede bulunan çalıştırılabilir alternatifler:")
        for p in found:
            print(" -", p)
        print("\nÖrnek:")
        print(" !python ilim-assistant/desktop_server.py")
    else:
        print("Çalıştırılabilir ana giriş dosyası bulunamadı. Repo yapısını kontrol edin.")